# Phase IV — Prompts 12–14
This notebook evaluates the existing simulated experiment. Upload or locate your actual `multi_seed_results.csv` before running analysis. Do not treat the historical summary values in the report as verified until this notebook runs on raw data.

## Prompt 12 — Frozen protocol
Research question: How does conditional downstream error propagation affect stage-error accumulation in a four-stage simulated workflow? Recommended pathway: controlled comparison of independent vs propagation, followed by sensitivity analyses. Primary metric: mean stage errors. Secondary metric: any-stage-error rate with Wilson 95% interval.

In [ ]:
from pathlib import Path
import pandas as pd, numpy as np, matplotlib.pyplot as plt


## Prompt 13 — Validate and summarize actual CSV

In [ ]:
# Set this path to your actual uploaded/exported Phase III CSV.
INPUT_CSV = Path("predictions/multi_seed_results.csv")
if not INPUT_CSV.exists():
    raise FileNotFoundError(f"CSV not found: {INPUT_CSV.resolve()}. Upload the file or update INPUT_CSV.")
df = pd.read_csv(INPUT_CSV)
required = {"error_rate", "model", "stage_errors", "workflow_failed"}
assert required.issubset(df.columns), f"Missing columns: {required-set(df.columns)}"
assert not df[list(required - {"workflow_failed"})].isna().any().any()
df["error_rate"] = pd.to_numeric(df["error_rate"], errors="raise")
df["stage_errors"] = pd.to_numeric(df["stage_errors"], errors="raise")
assert df["error_rate"].between(0, 1).all()
assert (df["stage_errors"] >= 0).all()
df["workflow_failed"] = df["workflow_failed"].astype(str).str.lower().map({"true": True, "false": False, "1": True, "0": False})
assert df["workflow_failed"].notna().all()
assert (df["workflow_failed"] == (df["stage_errors"] > 0)).all()
keys = [c for c in ["trial", "error_rate", "model", "seed"] if c in df.columns]
assert not df.duplicated(keys).any(), "Duplicate trial keys detected"
summary = df.groupby(["error_rate", "model"]).agg(
    n=("stage_errors", "size"),
    mean_stage_errors=("stage_errors", "mean"),
    std_stage_errors=("stage_errors", "std"),
    failure_rate=("workflow_failed", "mean")
).reset_index()
def wilson(p, n, z=1.96):
    den = 1 + z*z/n
    center = (p + z*z/(2*n))/den
    half = z*np.sqrt(p*(1-p)/n + z*z/(4*n*n))/den
    return center-half, center+half
ci = [wilson(float(r.failure_rate), int(r.n)) for r in summary.itertuples()]
summary["failure_ci95_low"] = [x[0] for x in ci]
summary["failure_ci95_high"] = [x[1] for x in ci]
OUT = Path("results/phase_iv")
OUT.mkdir(parents=True, exist_ok=True)
summary.to_csv(OUT / "phaseiv_results.csv", index=False)
display(summary)
print(f"PASS: {len(df)} raw rows validated.")


## Prompt 14 — Plots and interpretation guardrails

In [ ]:
for metric, ylabel, title, filename in [
    ("mean_stage_errors", "Mean stage errors per workflow", "Stage-error accumulation", "phaseiv_stage_errors.png"),
    ("failure_rate", "Workflow failure rate", "Any-stage-error failure rate", "phaseiv_failure_rate.png")
]:
    plt.figure(figsize=(7, 4.5))
    for model in summary["model"].unique():
        d = summary[summary.model == model]
        plt.plot(d.error_rate, d[metric], marker="o", label=model)
    plt.xlabel("Base per-stage error probability")
    plt.ylabel(ylabel); plt.title(title); plt.grid(True, alpha=.3); plt.legend(); plt.tight_layout()
    plt.savefig(OUT / filename, dpi=200); plt.show()


**Interpretation guardrails:** The simulation can support claims only about its specified error process. If workflow failure means any stage error, downstream propagation can raise stage-error counts without changing the binary failure indicator. Do not infer real LLM-agent reliability, manuscript correctness, citation validity, novelty, or statistical significance. Follow-up: vary propagation increment, vary stage count, and add a verification intervention plus terminal correctness.